# Solutions 5 · Predict RUL from Selected Sensors

Quiz: choose the right sensors, then predict the remaining useful life (RUL) of a machine with a linear model.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/notebooks/sol-05-predictive-maintenance.ipynb)

- Lecture: [Chapter 5 · Predictive Maintenance with Linear Models](https://github.com/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/en-05-predictive-maintainance.md)
- Lab: [LM501 · RUL from Sensors](https://rathachai.github.io/DA-LAB/learnings/linear/lm501.html)
- Demo notebook: [nb-05-predictive-maintenance](https://colab.research.google.com/github/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/notebooks/nb-05-predictive-maintenance.ipynb)

> Powered by: Sonnet  
> AI Whisperer: [rathachai.github.io](https://rathachai.github.io)

---

## How to use
This notebook contains **sample solutions**. Try the quiz notebook first, then compare. Run the SETUP cell first, then the cells in order. Your numbers may differ slightly if you choose other features or seeds.

notebook นี้มี **ตัวอย่างเฉลย** ให้ลองทำ notebook แบบทดสอบก่อน แล้วจึงนำมาเปรียบเทียบ รันเซลล์ SETUP ก่อน จากนั้นรันเซลล์อื่นตามลำดับ ตัวเลขของคุณอาจต่างเล็กน้อยหากเลือกฟีเจอร์หรือ seed อื่น

## SETUP
The quiz data are in `quiz05_sensors.csv`: **1,000 snapshots of 40 machines** (25 snapshots per machine). The columns are `id`, `machine`, six sensors `s1`–`s6` and the target `y` = **remaining useful life (RUL) in hours**. Not every sensor is useful.

ข้อมูลของแบบทดสอบอยู่ในไฟล์ `quiz05_sensors.csv`: **ภาพบันทึก 1,000 ครั้งจากเครื่องจักร 40 เครื่อง** (เครื่องละ 25 ครั้ง) คอลัมน์ได้แก่ `id`, `machine`, เซนเซอร์ 6 ตัว `s1`–`s6` และตัวแปรเป้าหมาย `y` = **อายุการใช้งานที่เหลือ (RUL) หน่วยเป็นชั่วโมง** เซนเซอร์บางตัวไม่มีประโยชน์

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score

url = "https://rathachai.github.io/DA-LAB/learnings/linear/data/quiz05_sensors.csv"
df = pd.read_csv(url)
print(df.shape)
df

### Exercise 1 · Look at the data

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 1.** Display `df.describe()` and the number of snapshots per machine. Then compute the **correlation of every sensor with `y`** and draw them as a bar chart. Which sensors look useful, and which look useless?<br>
**Also plot** `s3` against `y` and `s4` against `y` (scatter). What do you notice that the correlation number alone does not tell you?<br>
*Hint: `df.describe()`, `df.groupby("machine").size()`, `df[sensor_columns].corrwith(df["y"])`, `.plot.bar()`, `plt.scatter`.*

**แบบฝึกหัด 1.** แสดง `df.describe()` และจำนวนภาพบันทึกต่อเครื่องจักร จากนั้นคำนวณ **สหสัมพันธ์ (correlation) ของเซนเซอร์ทุกตัวกับ `y`** และวาดเป็นแผนภูมิแท่ง เซนเซอร์ตัวใดดูมีประโยชน์ และตัวใดดูไม่มีประโยชน์<br>
**วาดเพิ่ม** `s3` เทียบกับ `y` และ `s4` เทียบกับ `y` (scatter) สังเกตเห็นอะไรที่ค่าสหสัมพันธ์เพียงอย่างเดียวบอกไม่ได้<br>
*คำใบ้: `df.describe()`, `df.groupby("machine").size()`, `df[sensor_columns].corrwith(df["y"])`, `.plot.bar()`, `plt.scatter`*

</div>

In [ ]:
print(df.describe().round(2))
print(df.groupby("machine").size().describe())

sensors = ["s1", "s2", "s3", "s4", "s5", "s6"]
corr = df[sensors].corrwith(df["y"])
print(corr.round(3))
corr.plot.bar(color=["#2563eb" if v > 0 else "#dc2626" for v in corr], title="Correlation with y (RUL)")
plt.axhline(0, color="k", lw=0.8); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].scatter(df["s3"], df["y"], s=6); ax[0].set(xlabel="s3", ylabel="y (RUL)")
ax[1].scatter(df["s4"], df["y"], s=6); ax[1].set(xlabel="s4", ylabel="y (RUL)")
plt.tight_layout(); plt.show()

<div style="border:2px solid #16a34a; border-radius:8px; padding:12px; background:#f0fdf4">

**Solution notes.** `s1` (r = -0.90), `s4` (r = -0.81), `s2` (-0.70) and `s6` (+0.63) are useful; `s5` (+0.10) and `s3` (-0.00) look useless by r. But the scatter shows that `s3` is **U-shaped** against `y` (a straight-line r cannot see it) and `s4` is **curved** (big changes at small values), so a logarithm should straighten it.

**บันทึกเฉลย.** `s1` (r = -0.90), `s4` (r = -0.81), `s2` (-0.70) และ `s6` (+0.63) มีประโยชน์ ส่วน `s5` (+0.10) และ `s3` (-0.00) ดูไม่มีประโยชน์เมื่อดูจากค่า r แต่กราฟ scatter แสดงว่า `s3` มีรูปร่างเป็น **ตัวยู (U)** เมื่อเทียบกับ `y` (ค่า r ซึ่งวัดแนวเส้นตรงมองไม่เห็น) และ `s4` มี **ความโค้ง** (เปลี่ยนมากเมื่อค่าน้อย) ดังนั้นการใช้ลอการิทึมน่าจะช่วยทำให้เป็นเส้นตรงขึ้น

</div>

### Exercise 2 · Process the data and select X and y

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 2.** Add a column `ln_s4 = np.log(df["s4"])` and compare its correlation with `y` to the one of `s4`. Then **select your features** `X` (a list of columns) and the target `y`. Explain in one line why you keep or drop each sensor.<br>
*Hint: `np.log`, `df[[...]]`. Remember that `id` and `machine` are labels, not sensors.*

**แบบฝึกหัด 2.** เพิ่มคอลัมน์ `ln_s4 = np.log(df["s4"])` และเปรียบเทียบค่าสหสัมพันธ์กับ `y` ของคอลัมน์นี้กับของ `s4` จากนั้น **เลือกฟีเจอร์** `X` (รายชื่อคอลัมน์) และตัวแปรเป้าหมาย `y` อธิบายใน 1 บรรทัดว่าทำไมจึงเก็บหรือตัดเซนเซอร์แต่ละตัว<br>
*คำใบ้: `np.log`, `df[[...]]` จำไว้ว่า `id` และ `machine` เป็นป้ายชื่อ ไม่ใช่เซนเซอร์*

</div>

In [ ]:
df["ln_s4"] = np.log(df["s4"])
print("corr s4    :", round(df["s4"].corr(df["y"]), 3))
print("corr ln_s4 :", round(df["ln_s4"].corr(df["y"]), 3))

features = ["s1", "s2", "s6", "ln_s4"]     # s3 (U-shape), s5 (noise), raw s4 (replaced by ln_s4) are dropped
X = df[features]
y = df["y"]
X

<div style="border:2px solid #16a34a; border-radius:8px; padding:12px; background:#f0fdf4">

**Solution notes.** `ln_s4` has r = -0.913 against -0.812 for raw `s4`, so the log helps. Selected: `s1`, `s2`, `s6`, `ln_s4`. Dropped: `s3` (U-shape, no linear relation), `s5` (noise), raw `s4` (replaced by its log), `id` and `machine` (labels).

**บันทึกเฉลย.** `ln_s4` มี r = -0.913 เทียบกับ -0.812 ของ `s4` ดิบ ดังนั้นลอการิทึมช่วยได้ ฟีเจอร์ที่เลือก: `s1`, `s2`, `s6`, `ln_s4` ฟีเจอร์ที่ตัดออก: `s3` (รูปตัวยู ไม่มีความสัมพันธ์เชิงเส้น), `s5` (สัญญาณรบกวน), `s4` ดิบ (ถูกแทนด้วยค่าลอการิทึม), `id` และ `machine` (ป้ายชื่อ)

</div>

### Exercise 3 · Split by machine, train and predict

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 3.** Split the data into train and test **by machine** (about 30 % of the machines in the test set; use `random_state=1`). Fit a `LinearRegression` on your selected features using the train part only, and predict `y_pred` for the test part. Print the equation (intercept and one coefficient per feature).<br>
*Hint: `GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=1).split(X, y, groups=df["machine"])` returns train and test row positions.*

**แบบฝึกหัด 3.** แบ่งข้อมูลเป็นชุดฝึก (train) และชุดทดสอบ (test) **ตามเครื่องจักร** (ประมาณ 30 % ของเครื่องจักรอยู่ในชุดทดสอบ ใช้ `random_state=1`) ฝึก `LinearRegression` ด้วยฟีเจอร์ที่เลือกโดยใช้เฉพาะชุดฝึก แล้วทำนาย `y_pred` ของชุดทดสอบ พิมพ์สมการ (ค่าตัดแกน และสัมประสิทธิ์ของแต่ละฟีเจอร์)<br>
*คำใบ้: `GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=1).split(X, y, groups=df["machine"])` จะคืนตำแหน่งแถวของชุดฝึกและชุดทดสอบ*

</div>

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=1)
tr, te = next(gss.split(X, y, groups=df["machine"]))
X_train, X_test, y_train, y_test = X.iloc[tr], X.iloc[te], y.iloc[tr], y.iloc[te]
print("train machines:", df.iloc[tr]["machine"].nunique(), " test machines:", df.iloc[te]["machine"].nunique())

model = LinearRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)
print("y_pred = %.2f" % model.intercept_ + "".join(f" {c:+.2f}*{n}" for c, n in zip(model.coef_, features)))

<div style="border:2px solid #16a34a; border-radius:8px; padding:12px; background:#f0fdf4">

**Solution notes.** The split puts 28 machines in train and 12 in test, so the model is tested on machines it has never seen. The equation is roughly RUL = 457 - 32.8 s1 - 1.1 s2 + 0.5 s6 - 60.4 ln(s4).

**บันทึกเฉลย.** การแบ่งข้อมูลทำให้ชุดฝึกมี 28 เครื่อง และชุดทดสอบมี 12 เครื่อง ดังนั้นโมเดลถูกทดสอบกับเครื่องจักรที่ไม่เคยเห็นมาก่อน สมการโดยประมาณคือ RUL = 457 - 32.8 s1 - 1.1 s2 + 0.5 s6 - 60.4 ln(s4)

</div>

### Exercise 4 · Evaluate and visualize

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 4.** Report **MAE, RMSE, MAPE and R²** for the train part and for the test part in one small DataFrame. Then plot `y` (actual) against `y_pred` for the test part with the diagonal line `y = y_pred`. Where does the model do worst?<br>
*Hint: `mean_absolute_error`, `mean_squared_error(..., squared=False)` or `np.sqrt`, `mean_absolute_percentage_error`, `r2_score`; `plt.plot([0, 450], [0, 450])` for the diagonal.*

**แบบฝึกหัด 4.** รายงานค่า **MAE, RMSE, MAPE และ R²** ของชุดฝึกและชุดทดสอบใน DataFrame ขนาดเล็กตารางเดียว จากนั้นวาดกราฟ `y` (ค่าจริง) เทียบกับ `y_pred` ของชุดทดสอบ พร้อมเส้นทแยงมุม `y = y_pred` โมเดลทำงานได้แย่ที่สุดตรงไหน<br>
*คำใบ้: `mean_absolute_error`, `mean_squared_error(..., squared=False)` หรือ `np.sqrt`, `mean_absolute_percentage_error`, `r2_score`; ใช้ `plt.plot([0, 450], [0, 450])` วาดเส้นทแยงมุม*

</div>

In [ ]:
y_train_pred = model.predict(X_train)

mae = mean_absolute_error(y_train, y_train_pred)
print("train MAE  =", round(mae, 3))

rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
print("train RMSE =", round(rmse, 3))

mape = mean_absolute_percentage_error(y_train, y_train_pred)
print("train MAPE =", round(mape, 3))

r2 = r2_score(y_train, y_train_pred)
print("train R2   =", round(r2, 3))

mae = mean_absolute_error(y_test, y_pred)
print("test  MAE  =", round(mae, 3))

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("test  RMSE =", round(rmse, 3))

mape = mean_absolute_percentage_error(y_test, y_pred)
print("test  MAPE =", round(mape, 3))

r2 = r2_score(y_test, y_pred)
print("test  R2   =", round(r2, 3))

In [ ]:
plt.scatter(y_test, y_pred, s=8, alpha=0.6)
plt.plot([0, 450], [0, 450], "k--", label="y = y_pred")
plt.xlabel("actual RUL (y)"); plt.ylabel("predicted RUL (y_pred)"); plt.legend(); plt.show()

<div style="border:2px solid #16a34a; border-radius:8px; padding:12px; background:#f0fdf4">

**Solution notes.** Train: MAE 20.9, R2 0.93. Test: MAE 22.6, RMSE 28.7, R2 0.88. The test error is only a little higher, so the model generalises. MAPE is large (0.37) because RUL near zero makes small absolute errors big in percent. The scatter is widest at large RUL (the model is least sure about machines that are far from failure).

**บันทึกเฉลย.** ชุดฝึก: MAE 20.9, R2 0.93 ชุดทดสอบ: MAE 22.6, RMSE 28.7, R2 0.88 ค่าความคลาดเคลื่อนของชุดทดสอบสูงกว่าเพียงเล็กน้อย แสดงว่าโมเดลใช้งานกับข้อมูลใหม่ได้ดี (generalise) MAPE มีค่าสูง (0.37) เพราะเมื่อ RUL ใกล้ศูนย์ ความคลาดเคลื่อนสัมบูรณ์เพียงเล็กน้อยก็กลายเป็นเปอร์เซ็นต์ที่มาก กราฟ scatter กระจายกว้างที่สุดเมื่อ RUL มาก (โมเดลไม่แน่ใจที่สุดกับเครื่องจักรที่ยังอยู่ห่างจากการเสีย)

</div>

### Exercise 5 · Does feature selection help?

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 5.** Repeat Exercises 3–4 for three feature sets: (a) all raw sensors `s1`–`s6`, (b) your selected features, (c) only the single best sensor. Repeat each with 20 different random splits by machine and report the **mean test MAE** of each set in a table.<br>
*Hint: loop over `seed in range(20)` with `GroupShuffleSplit(test_size=0.3, random_state=seed)`; store the results in a list of dictionaries and build a `pd.DataFrame`.*

**แบบฝึกหัด 5.** ทำแบบฝึกหัด 3–4 ซ้ำกับฟีเจอร์ 3 ชุด: (a) เซนเซอร์ดิบทั้งหมด `s1`–`s6`, (b) ฟีเจอร์ที่คุณเลือก, (c) เซนเซอร์ที่ดีที่สุดเพียงตัวเดียว ทำซ้ำแต่ละชุดด้วยการแบ่งแบบสุ่มตามเครื่องจักร 20 ครั้ง และรายงาน **ค่า MAE เฉลี่ยของชุดทดสอบ** ของแต่ละชุดในตาราง<br>
*คำใบ้: วนลูป `seed in range(20)` กับ `GroupShuffleSplit(test_size=0.3, random_state=seed)` เก็บผลลัพธ์เป็นลิสต์ของ dictionary แล้วสร้าง `pd.DataFrame`*

</div>

In [ ]:
sets = {"(a) all raw s1-s6": ["s1", "s2", "s3", "s4", "s5", "s6"],
        "(b) selected": features,
        "(c) best single (s1)": ["s1"]}
rows = []
for seed in range(20):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed).split(df, df["y"], groups=df["machine"]))
    for name, cols in sets.items():
        m = LinearRegression().fit(df.iloc[tr][cols], df.iloc[tr]["y"])
        rows.append({"set": name, "test_MAE": mean_absolute_error(df.iloc[te]["y"], m.predict(df.iloc[te][cols]))})
pd.DataFrame(rows).groupby("set")["test_MAE"].agg(["mean", "std"]).round(2)

<div style="border:2px solid #16a34a; border-radius:8px; padding:12px; background:#f0fdf4">

**Solution notes.** Mean test MAE over 20 splits: all raw sensors 24.5, **selected 21.6**, best single sensor 33.7. Selecting and transforming features beats using everything, and one sensor alone is clearly not enough.

**บันทึกเฉลย.** ค่า MAE เฉลี่ยของชุดทดสอบจาก 20 การแบ่ง: เซนเซอร์ดิบทั้งหมด 24.5, **ชุดที่เลือก 21.6**, เซนเซอร์เดี่ยวที่ดีที่สุด 33.7 การเลือกและแปลงฟีเจอร์ให้ผลดีกว่าการใช้ทั้งหมด และเซนเซอร์ตัวเดียวไม่เพียงพออย่างชัดเจน

</div>

### Exercise 6 · Conclusion

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 6.** In three or four sentences: which sensors did you select and why? How much better is the selected set than "all sensors" and than the best single sensor? What would happen to the test error if you split by **row** instead of by machine?

**แบบฝึกหัด 6.** เขียน 3–4 ประโยค: คุณเลือกเซนเซอร์ตัวใดและเพราะอะไร ชุดที่เลือกดีกว่า "เซนเซอร์ทั้งหมด" และดีกว่าเซนเซอร์ที่ดีที่สุดเพียงตัวเดียวมากเท่าใด ค่าความคลาดเคลื่อนของชุดทดสอบจะเป็นอย่างไรถ้าแบ่งข้อมูลตาม **แถว** แทนการแบ่งตามเครื่องจักร

</div>

<div style="border:2px solid #16a34a; border-radius:8px; padding:12px; background:#f0fdf4">

**Your answer:** I selected `s1`, `s2`, `s6` and `ln_s4` because they correlate with RUL (|r| 0.63-0.91) and the log straightens `s4`; `s3` (U-shape) and `s5` (noise) were dropped. The selected set gives test MAE about 21.6 h against 24.5 h for all raw sensors (about 12 % better) and 33.7 h for the best single sensor (about 36 % better). If I split by **row**, snapshots of the same machine would appear in both train and test, so the model could recognise machines it has seen and the test error would look better than it really is for a new machine.

**คำตอบของคุณ:** ฉันเลือก `s1`, `s2`, `s6` และ `ln_s4` เพราะสัมพันธ์กับ RUL (|r| 0.63-0.91) และลอการิทึมช่วยทำให้ `s4` เป็นเส้นตรงขึ้น ส่วน `s3` (รูปตัวยู) และ `s5` (สัญญาณรบกวน) ถูกตัดออก ชุดที่เลือกให้ MAE ของชุดทดสอบประมาณ 21.6 ชม. เทียบกับ 24.5 ชม. ของเซนเซอร์ดิบทั้งหมด (ดีกว่าประมาณ 12 %) และ 33.7 ชม. ของเซนเซอร์เดี่ยวที่ดีที่สุด (ดีกว่าประมาณ 36 %) ถ้าแบ่งตาม **แถว** ภาพบันทึกของเครื่องจักรเครื่องเดียวกันจะปรากฏทั้งในชุดฝึกและชุดทดสอบ โมเดลจึงอาจจำเครื่องที่เคยเห็นได้ และค่าความคลาดเคลื่อนของชุดทดสอบจะดูดีกว่าความเป็นจริงสำหรับเครื่องจักรเครื่องใหม่

</div>

## Done
Compare your answers with the demo notebook and discuss them in class.

เปรียบเทียบคำตอบกับ notebook สาธิต แล้วอภิปรายร่วมกันในชั้นเรียน

> Powered by: Sonnet  
> AI Whisperer: [rathachai.github.io](https://rathachai.github.io)

---